# **Project Name** - PhonePe Transaction Insights

##### **Project Type** - EDA
##### **Contribution** - Individual
##### **Name -** Kshitij Jadhav

# **Project Summary**

PhonePe has grown into one of India's largest digital payment platforms, processing billions of transactions every year. This project digs into the publicly available PhonePe Pulse dataset — which covers aggregated transaction, user, and insurance data across every state, district, and pin code in India from 2018 to 2023.

The dataset is structured in a way that separates data into three broad buckets: **Aggregated** (state-level yearly/quarterly rollups), **Map** (district-level breakdowns), and **Top** (top-performing districts and pin codes). Each bucket further splits into transaction, user, and insurance categories, giving a fairly complete picture of how digital payments have evolved across the country.

The main goal here is to understand the growth pattern of UPI transactions over time, find out which states and transaction types dominate the volume, figure out what devices people are using, and surface any interesting regional patterns. Along the way I've built 20+ charts covering univariate, bivariate, and multivariate angles.

A few things I found interesting during the analysis: Telangana, Maharashtra, and Karnataka consistently rank among the top states for both transaction count and value. Peer-to-peer payments (P2P) dominate the transaction type mix by a wide margin. Android device adoption is heavily skewed toward a handful of brands. And while transaction volumes have grown every single year, the per-transaction average has been declining — suggesting that smaller everyday payments are being brought onto UPI rather than just large transfers.

The final outputs include the EDA notebook you are reading now, the SQL database with all tables populated, and a Streamlit dashboard that lets you explore the data interactively by state, quarter, and category.


# **GitHub Link**

https://github.com/kshitij-create/PhonePe-Transaction-Insights-EDA


# **Problem Statement**

Digital payment adoption in India has exploded over the last five years, but raw growth numbers don't tell the whole story. Which states are truly leading adoption versus just riding population size? Which transaction categories are driving volume? Are users moving away from merchant payments back to P2P? How is insurance penetration growing through UPI?

This project tries to answer these questions using the PhonePe Pulse dataset.

#### **Business Objective**

The core objective is to give a product or business team at PhonePe a clear view of:
1. Where transaction volume is concentrated geographically
2. Which payment categories are growing fastest
3. What the user base looks like in terms of devices and engagement
4. Which districts and pin codes represent untapped or high-growth markets

These insights can directly feed into marketing spend allocation, feature prioritization, and partnership decisions with merchants in high-potential regions.


# **General Guidelines**

1. Code is structured, commented, and follows PEP8 roughly.
2. Exception handling is added where file loading might fail.
3. Each chart has a reason, insight, and business impact note below it.
4. At least 20 charts covering univariate, bivariate, and multivariate analysis.

---

# Let's Begin!


## 1. Know Your Data

### Import Libraries

In [ ]:
# standard stuff first
import os
import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

# making plots look a bit nicer
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11

print("Libraries loaded successfully")


### Dataset Loading

In [ ]:
# Clone the PhonePe Pulse repo if it doesn't already exist
import subprocess

if not os.path.exists('pulse'):
    print("Cloning PhonePe Pulse repo... this takes a minute")
    subprocess.run(['git', 'clone', '--depth=1',
                    'https://github.com/PhonePe/pulse.git'], check=True)
    print("Done.")
else:
    print("Repo already cloned, skipping.")

BASE = 'pulse/data'
print("Base data path:", BASE)


In [ ]:
# helper to recursively load all JSON files under a given path
# returns a flat list of dicts with state, year, quarter, and payload

def load_json_files(folder_path):
    records = []
    for root, dirs, files in os.walk(folder_path):
        for fname in files:
            if not fname.endswith('.json'):
                continue
            fpath = os.path.join(root, fname)
            parts = fpath.replace('\\', '/').split('/')
            try:
                # typical path: .../state/<state_name>/<year>/<quarter>.json
                state = parts[-3]
                year  = int(parts[-2])
                quarter = int(fname.replace('.json', ''))
                with open(fpath, 'r') as f:
                    data = json.load(f)
                records.append({
                    'state': state,
                    'year': year,
                    'quarter': quarter,
                    'data': data
                })
            except Exception:
                # some paths don't follow the state/year/quarter pattern
                pass
    return records


In [ ]:
# ─── AGGREGATED TRANSACTION ─────────────────────────────────────────
agg_txn_path = os.path.join(BASE, 'aggregated', 'transaction', 'country', 'india', 'state')
raw_agg_txn  = load_json_files(agg_txn_path)

agg_txn_rows = []
for rec in raw_agg_txn:
    txn_data = rec['data'].get('data', {}).get('transactionData', [])
    for item in txn_data:
        agg_txn_rows.append({
            'state'           : rec['state'],
            'year'            : rec['year'],
            'quarter'         : rec['quarter'],
            'transaction_type': item['name'],
            'count'           : item['paymentInstruments'][0]['count'],
            'amount'          : item['paymentInstruments'][0]['amount']
        })

df_agg_txn = pd.DataFrame(agg_txn_rows)
print("Aggregated Transaction shape:", df_agg_txn.shape)
df_agg_txn.head()


In [ ]:
# ─── AGGREGATED USER ────────────────────────────────────────────────
agg_user_path = os.path.join(BASE, 'aggregated', 'user', 'country', 'india', 'state')
raw_agg_user  = load_json_files(agg_user_path)

agg_user_rows = []
for rec in raw_agg_user:
    payload = rec['data'].get('data', {})
    reg_users  = payload.get('aggregated', {}).get('registeredUsers', 0)
    app_opens  = payload.get('aggregated', {}).get('appOpens', 0)
    user_by_device = payload.get('usersByDevice', []) or []
    for dev in user_by_device:
        agg_user_rows.append({
            'state'         : rec['state'],
            'year'          : rec['year'],
            'quarter'       : rec['quarter'],
            'registered_users': reg_users,
            'app_opens'     : app_opens,
            'brand'         : dev.get('brand', 'Unknown'),
            'device_count'  : dev.get('count', 0),
            'device_pct'    : dev.get('percentage', 0)
        })

df_agg_user = pd.DataFrame(agg_user_rows)
print("Aggregated User shape:", df_agg_user.shape)
df_agg_user.head()


In [ ]:
# ─── AGGREGATED INSURANCE ───────────────────────────────────────────
agg_ins_path = os.path.join(BASE, 'aggregated', 'insurance', 'country', 'india', 'state')
raw_agg_ins  = load_json_files(agg_ins_path)

agg_ins_rows = []
for rec in raw_agg_ins:
    txn_data = rec['data'].get('data', {}).get('transactionData', [])
    for item in txn_data:
        agg_ins_rows.append({
            'state'  : rec['state'],
            'year'   : rec['year'],
            'quarter': rec['quarter'],
            'count'  : item['paymentInstruments'][0]['count'],
            'amount' : item['paymentInstruments'][0]['amount']
        })

df_agg_ins = pd.DataFrame(agg_ins_rows)
print("Aggregated Insurance shape:", df_agg_ins.shape)
df_agg_ins.head()


In [ ]:
# ─── MAP TRANSACTION ────────────────────────────────────────────────
map_txn_path = os.path.join(BASE, 'map', 'transaction', 'hover', 'country', 'india', 'state')
raw_map_txn  = load_json_files(map_txn_path)

map_txn_rows = []
for rec in raw_map_txn:
    hover_data = rec['data'].get('data', {}).get('hoverDataList', [])
    for item in hover_data:
        map_txn_rows.append({
            'state'   : rec['state'],
            'year'    : rec['year'],
            'quarter' : rec['quarter'],
            'district': item.get('name', ''),
            'count'   : item.get('metric', [{}])[0].get('count', 0),
            'amount'  : item.get('metric', [{}])[0].get('amount', 0)
        })

df_map_txn = pd.DataFrame(map_txn_rows)
print("Map Transaction shape:", df_map_txn.shape)
df_map_txn.head()


In [ ]:
# ─── MAP USER ───────────────────────────────────────────────────────
map_user_path = os.path.join(BASE, 'map', 'user', 'hover', 'country', 'india', 'state')
raw_map_user  = load_json_files(map_user_path)

map_user_rows = []
for rec in raw_map_user:
    hover_data = rec['data'].get('data', {}).get('hoverData', {})
    for district, info in hover_data.items():
        map_user_rows.append({
            'state'            : rec['state'],
            'year'             : rec['year'],
            'quarter'          : rec['quarter'],
            'district'         : district,
            'registered_users' : info.get('registeredUsers', 0),
            'app_opens'        : info.get('appOpens', 0)
        })

df_map_user = pd.DataFrame(map_user_rows)
print("Map User shape:", df_map_user.shape)
df_map_user.head()


In [ ]:
# ─── TOP TRANSACTION ────────────────────────────────────────────────
top_txn_path = os.path.join(BASE, 'top', 'transaction', 'country', 'india', 'state')
raw_top_txn  = load_json_files(top_txn_path)

top_txn_rows = []
for rec in raw_top_txn:
    top_data = rec['data'].get('data', {})
    for level in ['districts', 'pincodes']:
        for item in (top_data.get(level) or []):
            top_txn_rows.append({
                'state'     : rec['state'],
                'year'      : rec['year'],
                'quarter'   : rec['quarter'],
                'level'     : level,
                'entity'    : item.get('entityName', ''),
                'count'     : item.get('metric', {}).get('count', 0),
                'amount'    : item.get('metric', {}).get('amount', 0)
            })

df_top_txn = pd.DataFrame(top_txn_rows)
print("Top Transaction shape:", df_top_txn.shape)
df_top_txn.head()


In [ ]:
# ─── TOP USER ───────────────────────────────────────────────────────
top_user_path = os.path.join(BASE, 'top', 'user', 'country', 'india', 'state')
raw_top_user  = load_json_files(top_user_path)

top_user_rows = []
for rec in raw_top_user:
    top_data = rec['data'].get('data', {})
    for level in ['districts', 'pincodes']:
        for item in (top_data.get(level) or []):
            top_user_rows.append({
                'state'            : rec['state'],
                'year'             : rec['year'],
                'quarter'          : rec['quarter'],
                'level'            : level,
                'entity'           : item.get('name', ''),
                'registered_users' : item.get('registeredUsers', 0)
            })

df_top_user = pd.DataFrame(top_user_rows)
print("Top User shape:", df_top_user.shape)
df_top_user.head()


### Dataset First View

In [ ]:
# taking a quick peek at the main aggregated transaction table
df_agg_txn.head(10)


### Dataset Rows & Columns Count

In [ ]:
print("=== Row & Column Counts ===")
datasets = {
    'Aggregated Transaction' : df_agg_txn,
    'Aggregated User'        : df_agg_user,
    'Aggregated Insurance'   : df_agg_ins,
    'Map Transaction'        : df_map_txn,
    'Map User'               : df_map_user,
    'Top Transaction'        : df_top_txn,
    'Top User'               : df_top_user,
}
for name, df in datasets.items():
    print(f"  {name}: {df.shape[0]:,} rows  x  {df.shape[1]} cols")


### Dataset Information

In [ ]:
# main table we'll use the most
df_agg_txn.info()


#### Duplicate Values

In [ ]:
print("Duplicate rows in each dataset:")
for name, df in datasets.items():
    dups = df.duplicated().sum()
    print(f"  {name}: {dups}")


#### Missing Values / Null Values

In [ ]:
print("Null value counts:")
for name, df in datasets.items():
    nulls = df.isnull().sum().sum()
    print(f"  {name}: {nulls} total nulls")


In [ ]:
# visualizing nulls for main table - usually there aren't many but good to confirm
import missingno as msno
msno.matrix(df_agg_txn, figsize=(10, 4))
plt.title("Missing value pattern - Aggregated Transaction", fontsize=13)
plt.tight_layout()
plt.show()


### What did you know about your dataset?

The PhonePe Pulse data is clean and well-structured. Practically no null values or duplicates since it's aggregated data that PhonePe publishes themselves. The main aggregated transaction table has transaction type, count, amount, and is broken down by state, year, and quarter. The granularity goes from state level all the way down to pin code level in the Top tables. The dataset covers 2018 to 2023, capturing UPI's full growth arc in India. One thing worth noting is that the insurance data only starts from 2020 Q1 onwards, so it has fewer quarters than the transaction tables.


## 2. Understanding Your Variables

In [ ]:
# columns in the main aggregated transaction table
print("Aggregated Transaction columns:")
print(df_agg_txn.columns.tolist())
print()
print("Aggregated User columns:")
print(df_agg_user.columns.tolist())
print()
print("Map Transaction columns:")
print(df_map_txn.columns.tolist())


In [ ]:
# descriptive stats
df_agg_txn[['count', 'amount']].describe().applymap(lambda x: f"{x:,.2f}")


### Variables Description

**Aggregated Transaction table:**
- `state` — state name in lowercase-hyphenated format (e.g., 'andhra-pradesh')
- `year` — calendar year (2018–2023)
- `quarter` — quarter number (1 = Jan-Mar, 2 = Apr-Jun, etc.)
- `transaction_type` — one of: Recharge & bill payments, Peer-to-peer payments, Merchant payments, Financial Services, Others
- `count` — number of transactions of that type in that state/year/quarter
- `amount` — total transaction value in INR

**Aggregated User table:**
- `registered_users` — total users registered on PhonePe in that state/quarter
- `app_opens` — number of times the app was opened
- `brand` — device manufacturer (Samsung, Xiaomi, Vivo, etc.)
- `device_count` — users on that device brand
- `device_pct` — share of that brand among all users in that state/quarter

**Map Transaction / Map User:** Same fields but at district level instead of state.

**Top Transaction / Top User:** Only top-performing districts and pin codes — useful for spotting high-density pockets.


In [ ]:
# checking unique values for important categorical columns
print("Unique states:", df_agg_txn['state'].nunique())
print("Years covered:", sorted(df_agg_txn['year'].unique()))
print("Quarters:", sorted(df_agg_txn['quarter'].unique()))
print()
print("Transaction types:")
for t in df_agg_txn['transaction_type'].unique():
    print("  -", t)


## 3. Data Wrangling

In [ ]:
# ── 1. Fixing state names - they come as 'andhra-pradesh', make them readable
def clean_state(s):
    return s.replace('-', ' ').title()

for df in [df_agg_txn, df_agg_user, df_agg_ins, df_map_txn, df_map_user, df_top_txn, df_top_user]:
    if 'state' in df.columns:
        df['state'] = df['state'].apply(clean_state)

# ── 2. Adding a year_quarter string column - handy for time-series plots
for df in [df_agg_txn, df_agg_user, df_agg_ins]:
    df['year_quarter'] = df['year'].astype(str) + ' Q' + df['quarter'].astype(str)

# ── 3. Amount in crores (easier to read on charts)
for df in [df_agg_txn, df_agg_ins, df_map_txn]:
    df['amount_cr'] = (df['amount'] / 1e7).round(2)

# ── 4. Converting count to lakhs for cleaner axis labels
df_agg_txn['count_lakh'] = (df_agg_txn['count'] / 1e5).round(2)

print("Wrangling done.")
df_agg_txn.head(3)


In [ ]:
# ── 5. Building a state-level summary table - will use this for many charts
state_summary = (df_agg_txn
    .groupby('state', as_index=False)
    .agg(total_count=('count', 'sum'), total_amount=('amount', 'sum'))
    .sort_values('total_amount', ascending=False)
)
state_summary['total_amount_cr'] = (state_summary['total_amount'] / 1e7).round(1)
state_summary['total_count_lakh'] = (state_summary['total_count'] / 1e5).round(1)

# ── 6. Time series aggregated nationally
time_series = (df_agg_txn
    .groupby(['year', 'quarter', 'year_quarter'], as_index=False)
    .agg(total_count=('count', 'sum'), total_amount=('amount', 'sum'))
    .sort_values(['year', 'quarter'])
)
time_series['avg_txn_value'] = time_series['total_amount'] / time_series['total_count']

print("Summary tables created.")
print("State summary shape:", state_summary.shape)
print("Time series shape:", time_series.shape)


### What manipulations did I do and what did I find?

- **State name cleaning:** The raw JSON has states as 'andhra-pradesh' style strings. Converted to 'Andhra Pradesh' for readability on charts.
- **Year-Quarter column:** Created a combined string column (e.g., '2020 Q1') so time-series charts have clean x-axis labels.
- **Amount in Crores:** The raw amounts are in paise or rupees (very large numbers). Divided by 1e7 to get crores — much easier to read.
- **Aggregated summaries:** Built state-level and time-level rollup tables once, and reused them across charts rather than recomputing inside each plot.
- **No major data cleaning needed** since the PhonePe data is pre-aggregated and already clean.


## 4. Data Visualization, Storytelling & Experimenting with Charts

#### Chart - 1: Total Transaction Count by Year (Bar Chart)

In [ ]:
# Chart 1 - Bar chart: total transactions per year nationally
yearly = df_agg_txn.groupby('year')['count'].sum().reset_index()
yearly['count_cr'] = yearly['count'] / 1e7

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(yearly['year'].astype(str), yearly['count_cr'],
              color=sns.color_palette('Blues_d', len(yearly)), edgecolor='white', width=0.5)
ax.bar_label(bars, fmt='%.1fCr', padding=4, fontsize=10)
ax.set_xlabel('Year')
ax.set_ylabel('Transaction Count (Crores)')
ax.set_title('Total PhonePe Transactions per Year (National)', fontsize=13, fontweight='bold')
ax.set_ylim(0, yearly['count_cr'].max() * 1.15)
sns.despine()
plt.tight_layout()
plt.show()


**1. Why this chart?**  
A simple bar chart is the most direct way to show absolute volume growth year over year — no noise, just the headline number.

**2. Insights:**  
Transaction volume has grown at a staggering rate every single year. The jump from 2020 to 2021 is particularly steep, which aligns with the COVID-19 push toward contactless digital payments. Growth hasn't slowed down even after lockdowns ended.

**3. Business Impact:**  
Positive impact — consistent year-on-year growth confirms that the platform is healthy. The 2020 spike should be studied to understand what specifically drove onboarding so teams can replicate those conditions. No negative signals here.


#### Chart - 2: Quarterly Transaction Trend (Line Chart)

In [ ]:
# Chart 2 - Line chart: quarterly national transaction count trend
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(time_series['year_quarter'], time_series['total_count'] / 1e7,
        marker='o', linewidth=2, color='steelblue', markersize=5)
ax.fill_between(time_series['year_quarter'],
                time_series['total_count'] / 1e7, alpha=0.12, color='steelblue')
ax.set_xlabel('Year - Quarter')
ax.set_ylabel('Transaction Count (Crores)')
ax.set_title('Quarter-by-Quarter Transaction Volume (National)', fontsize=13, fontweight='bold')
plt.xticks(rotation=45, ha='right', fontsize=9)
sns.despine()
plt.tight_layout()
plt.show()


**1. Why this chart?**  
A line chart is the natural choice for time-series data. The area fill helps emphasize cumulative growth visually.

**2. Insights:**  
The trend is consistently upward with no single quarter showing a meaningful decline. Q4 (Oct-Dec) often shows a small uptick — probably driven by festive season purchases. The overall curve is nearly exponential up to 2022, then starts to smooth out a bit.

**3. Business Impact:**  
The festive-quarter pattern suggests that targeted promotions during Q4 could extract even more volume. No negative growth signals, but the flattening post-2022 is worth watching — it could mean market saturation is beginning in early-adopter states.


#### Chart - 3: Top 10 States by Transaction Amount (Horizontal Bar)

In [ ]:
# Chart 3 - Horizontal bar: top 10 states by total transaction value
top10_amount = state_summary.head(10).sort_values('total_amount_cr')

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(top10_amount['state'], top10_amount['total_amount_cr'],
               color=sns.color_palette('RdYlGn_r', 10), edgecolor='white')
ax.bar_label(bars, fmt='₹%.0f Cr', padding=5, fontsize=9)
ax.set_xlabel('Total Transaction Amount (₹ Crores)')
ax.set_title('Top 10 States by Transaction Value', fontsize=13, fontweight='bold')
sns.despine()
plt.tight_layout()
plt.show()


**1. Why this chart?**  
Horizontal bars are easier to read when state names are long. Color gradient from green to red emphasizes ranking.

**2. Insights:**  
Telangana, Maharashtra, and Karnataka dominate transaction value — all three are major metro-heavy states with high-income populations. Interestingly, Telangana (Hyderabad) often tops the list despite being smaller in population than Maharashtra, which points to very high per-capita digital payment adoption.

**3. Business Impact:**  
States like Uttar Pradesh and Bihar are massive by population but rank lower here — that's the opportunity. More merchant onboarding and UPI awareness campaigns in these states could unlock the next growth wave.


#### Chart - 4: Top 10 States by Transaction Count (Horizontal Bar)

In [ ]:
# Chart 4 - Top 10 states by transaction count
top10_count = state_summary.sort_values('total_count', ascending=False).head(10).sort_values('total_count')

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(top10_count['state'], top10_count['total_count_lakh'],
               color=sns.color_palette('Blues_d', 10), edgecolor='white')
ax.bar_label(bars, fmt='%.0f L', padding=5, fontsize=9)
ax.set_xlabel('Total Transaction Count (Lakhs)')
ax.set_title('Top 10 States by Transaction Count', fontsize=13, fontweight='bold')
sns.despine()
plt.tight_layout()
plt.show()


**1. Why this chart?**  
Separating count from amount is important — a state can have high count but low value (many small transactions) or vice versa.

**2. Insights:**  
Andhra Pradesh and Telangana together show very high counts relative to their size, suggesting that small-ticket everyday transactions (like grocery, auto-rickshaw payments) have fully penetrated these states. Maharashtra has high value but relatively lower count per capita.

**3. Business Impact:**  
High-count low-value states indicate that UPI has replaced cash for daily spending — a very sticky behavior. These states should be targeted for upsell products like savings features, credit lines, or insurance since users are already deeply engaged.


#### Chart - 5: Transaction Type Distribution (Pie Chart)

In [ ]:
# Chart 5 - Pie chart: breakdown by transaction type nationally
type_totals = df_agg_txn.groupby('transaction_type')['count'].sum().reset_index()
type_totals = type_totals.sort_values('count', ascending=False)

colors = sns.color_palette('pastel', len(type_totals))
fig, ax = plt.subplots(figsize=(8, 6))
wedges, texts, autotexts = ax.pie(
    type_totals['count'], labels=None,
    autopct='%1.1f%%', colors=colors,
    startangle=90, pctdistance=0.8,
    wedgeprops={'edgecolor': 'white', 'linewidth': 1.5}
)
ax.legend(wedges, type_totals['transaction_type'],
          title='Transaction Type', loc='center left', bbox_to_anchor=(1, 0, 0.5, 1))
ax.set_title('Transaction Type Mix (by Count)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


**1. Why this chart?**  
Pie chart works well here because we're showing part-to-whole composition across a small number of categories.

**2. Insights:**  
Peer-to-peer (P2P) payments take the largest slice by count, followed by merchant payments and recharge/bill payments. Financial Services is a tiny sliver, which makes sense as those are high-value but low-frequency.

**3. Business Impact:**  
P2P dominance means PhonePe's growth is partly driven by person-to-person transfers — this is great for virality (users invite others to pay them). However, merchant payments are the monetizable part for PhonePe's revenue model. Growing that slice should be a priority.


#### Chart - 6: Transaction Type Mix Over Years (Stacked Bar)

In [ ]:
# Chart 6 - Stacked bar: how transaction type mix changes year by year
type_year = df_agg_txn.groupby(['year', 'transaction_type'])['count'].sum().unstack(fill_value=0)
type_year_pct = type_year.div(type_year.sum(axis=1), axis=0) * 100  # normalized to %

colors = sns.color_palette('tab10', type_year_pct.shape[1])
ax = type_year_pct.plot(kind='bar', stacked=True, figsize=(11, 6),
                         color=colors, edgecolor='white', width=0.6)
ax.set_xlabel('Year')
ax.set_ylabel('Share of Transactions (%)')
ax.set_title('Transaction Type Share by Year', fontsize=13, fontweight='bold')
ax.legend(title='Type', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
plt.xticks(rotation=0)
sns.despine()
plt.tight_layout()
plt.show()


**1. Why this chart?**  
A normalized stacked bar shows composition shift over time — much more informative than raw counts when comparing mix across years.

**2. Insights:**  
Merchant payments have been steadily growing as a share of total transactions, which is a healthy sign for the ecosystem. The 'Others' category has shrunk significantly, suggesting the taxonomy has matured and transactions are now properly categorized.

**3. Business Impact:**  
The merchant payment share growth signals that more small businesses are accepting UPI — this is positive for PhonePe's revenue since interchange-based revenue comes from merchant transactions, not P2P.


#### Chart - 7: Average Transaction Value Trend (Line Chart)

In [ ]:
# Chart 7 - Average transaction value per quarter
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(time_series['year_quarter'], time_series['avg_txn_value'],
        marker='s', color='coral', linewidth=2, markersize=5)
ax.axhline(time_series['avg_txn_value'].mean(), ls='--', color='gray', linewidth=1, label='Overall average')
ax.set_xlabel('Year - Quarter')
ax.set_ylabel('Avg Transaction Value (₹)')
ax.set_title('Average Transaction Value per Quarter', fontsize=13, fontweight='bold')
ax.legend()
plt.xticks(rotation=45, ha='right', fontsize=9)
sns.despine()
plt.tight_layout()
plt.show()


**1. Why this chart?**  
Average transaction value (total amount / total count) is a derived metric that reveals behavioral shifts better than either count or amount alone.

**2. Insights:**  
The average transaction value has been declining over time. This is actually expected and healthy — as more users join, they start using UPI for smaller everyday expenses like chai, bus tickets, groceries. Early adopters used UPI more for larger bill payments and money transfers.

**3. Business Impact:**  
Declining average ticket size indicates mass market penetration — which is good for user growth but means per-transaction revenue (if any) decreases. Product teams should double down on higher-value services like mutual funds, insurance, credit to maintain ARPU.


#### Chart - 8: Top Device Brands Used (Bar Chart)

In [ ]:
# Chart 8 - Device brand distribution nationally
brand_totals = df_agg_user.groupby('brand')['device_count'].sum().reset_index()
brand_totals = brand_totals.sort_values('device_count', ascending=False).head(12)
brand_totals = brand_totals[brand_totals['brand'] != 'Unknown']  # drop unknowns

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.bar(brand_totals['brand'], brand_totals['device_count'] / 1e6,
              color=sns.color_palette('viridis', len(brand_totals)), edgecolor='white')
ax.bar_label(bars, fmt='%.1fM', padding=3, fontsize=9)
ax.set_xlabel('Device Brand')
ax.set_ylabel('User Count (Millions)')
ax.set_title('Top Device Brands on PhonePe', fontsize=13, fontweight='bold')
plt.xticks(rotation=30, ha='right')
sns.despine()
plt.tight_layout()
plt.show()


**1. Why this chart?**  
Bar chart works perfectly for ranking categorical data. This tells us the Android device landscape of PhonePe's user base.

**2. Insights:**  
Xiaomi, Samsung, and Vivo dominate — all three are budget-to-mid-range Android players. This reflects PhonePe's predominantly mass-market user base. Apple barely shows up, which makes sense since PhonePe's strongest UPI competition (GPay) has historically been stronger on iOS.

**3. Business Impact:**  
Knowing device distribution is critical for app optimization — the team should prioritize performance on Xiaomi/Samsung/Vivo since slow load times on these devices directly affect millions of users. Also, any Xiaomi-specific MIUI payment shortcuts could affect PhonePe's discoverability.


#### Chart - 9: Registered Users vs App Opens by State (Scatter)

In [ ]:
# Chart 9 - Scatter plot: registered users vs app opens — state level
user_state = (df_agg_user
    .groupby('state', as_index=False)
    .agg(total_users=('registered_users', 'sum'), total_opens=('app_opens', 'sum'))
)
user_state['opens_per_user'] = user_state['total_opens'] / user_state['total_users'].replace(0, np.nan)

fig, ax = plt.subplots(figsize=(10, 6))
scatter = ax.scatter(user_state['total_users'] / 1e6,
                     user_state['total_opens'] / 1e6,
                     c=user_state['opens_per_user'],
                     cmap='RdYlGn', s=80, alpha=0.8, edgecolors='white')
plt.colorbar(scatter, ax=ax, label='App Opens per User')

# label a few interesting states
for _, row in user_state.nlargest(5, 'total_users').iterrows():
    ax.annotate(row['state'], (row['total_users']/1e6, row['total_opens']/1e6),
                textcoords='offset points', xytext=(5, 4), fontsize=8)

ax.set_xlabel('Registered Users (Millions)')
ax.set_ylabel('App Opens (Millions)')
ax.set_title('Registered Users vs App Opens by State', fontsize=13, fontweight='bold')
sns.despine()
plt.tight_layout()
plt.show()


**1. Why this chart?**  
Scatter plot is the right tool for showing the relationship between two continuous variables. The color encoding adds a third dimension (engagement rate).

**2. Insights:**  
States in the upper-right corner (high users AND high opens) are the most engaged. Some states have large registered user bases but relatively fewer app opens — these are "registered but not active" states. The color shows that some mid-size states actually have higher opens-per-user ratios.

**3. Business Impact:**  
States with high registration but low opens are a churn risk. Re-engagement campaigns — push notifications, cashback incentives — should target these states specifically. Conversely, high-engagement states are ideal for premium product launches.


#### Chart - 10: Top 10 States by Registered Users (Bar)

In [ ]:
# Chart 10 - Registered users per state (top 10)
# collapsing to unique state-year-quarter values first to avoid double counting brand rows
user_state_reg = (df_agg_user
    .drop_duplicates(subset=['state', 'year', 'quarter'])
    .groupby('state', as_index=False)['registered_users']
    .sum()
    .sort_values('registered_users', ascending=False)
    .head(10)
)

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.bar(user_state_reg['state'], user_state_reg['registered_users'] / 1e6,
              color=sns.color_palette('crest', 10), edgecolor='white')
ax.bar_label(bars, fmt='%.1fM', padding=3, fontsize=9)
ax.set_xlabel('State')
ax.set_ylabel('Registered Users (Millions)')
ax.set_title('Top 10 States by Registered PhonePe Users', fontsize=13, fontweight='bold')
plt.xticks(rotation=30, ha='right')
sns.despine()
plt.tight_layout()
plt.show()


**1. Why this chart?**  
Simple bar chart to compare user base size across states — gives context to all the transaction analysis.

**2. Insights:**  
Maharashtra, Rajasthan, and Karnataka lead in registered users. The ranking isn't purely population-driven (e.g., UP has massive population but doesn't top this chart), confirming that smartphone penetration and digital literacy are key bottlenecks.

**3. Business Impact:**  
States with large populations but small user bases (UP, Bihar, MP) represent the biggest expansion opportunity. Vernacular language UX and feature phone support could help unlock these markets.


#### Chart - 11: Insurance Transaction Growth Over Time (Line)

In [ ]:
# Chart 11 - Insurance volume trend over quarters
ins_time = (df_agg_ins
    .groupby(['year', 'quarter', 'year_quarter'], as_index=False)
    .agg(total_count=('count', 'sum'), total_amount=('amount', 'sum'))
    .sort_values(['year', 'quarter'])
)

fig, ax1 = plt.subplots(figsize=(13, 5))
ax2 = ax1.twinx()

ax1.plot(ins_time['year_quarter'], ins_time['total_count'] / 1e5,
         color='steelblue', marker='o', linewidth=2, markersize=5, label='Count (Lakhs)')
ax2.plot(ins_time['year_quarter'], ins_time['total_amount'] / 1e7,
         color='coral', marker='s', linewidth=2, markersize=5, linestyle='--', label='Amount (₹Cr)')

ax1.set_xlabel('Year - Quarter')
ax1.set_ylabel('Insurance Txn Count (Lakhs)', color='steelblue')
ax2.set_ylabel('Insurance Txn Amount (₹ Crores)', color='coral')
ax1.set_title('Insurance Transactions on PhonePe Over Time', fontsize=13, fontweight='bold')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

plt.xticks(rotation=45, ha='right', fontsize=9)
sns.despine()
plt.tight_layout()
plt.show()


**1. Why this chart?**  
Dual-axis line chart lets us compare count and amount on different scales in one view, which is useful when the two metrics have very different magnitudes.

**2. Insights:**  
Insurance transactions started very small in 2020 and have grown rapidly, with especially sharp growth in 2022-23. The amount grows faster than the count — meaning premium amounts per transaction are increasing, not just more cheap policies.

**3. Business Impact:**  
Insurance is a high-margin vertical for PhonePe. The rapid growth here is a strong positive signal. The team should invest more in surfacing insurance products to users during relevant moments (e.g., after a large merchant payment, suggest health cover).


#### Chart - 12: Top 15 Districts by Transaction Amount (Bar)

In [ ]:
# Chart 12 - Top 15 districts nationally by total transaction amount
district_totals = (df_map_txn
    .groupby(['state', 'district'], as_index=False)
    .agg(total_amount=('amount', 'sum'), total_count=('count', 'sum'))
    .sort_values('total_amount', ascending=False)
    .head(15)
)
district_totals['label'] = district_totals['district'] + '\n(' + district_totals['state'].str[:5] + ')'

fig, ax = plt.subplots(figsize=(13, 5))
bars = ax.bar(district_totals['label'], district_totals['total_amount'] / 1e7,
              color=sns.color_palette('magma_r', 15), edgecolor='white')
ax.set_xlabel('District (State)')
ax.set_ylabel('Transaction Amount (₹ Crores)')
ax.set_title('Top 15 Districts by Total Transaction Value', fontsize=13, fontweight='bold')
plt.xticks(fontsize=8)
sns.despine()
plt.tight_layout()
plt.show()


**1. Why this chart?**  
Bar chart to rank districts — lets us go below state level and see which specific cities/districts drive the most value.

**2. Insights:**  
Hyderabad, Bengaluru, and Pune dominate district-level transaction values. These are all large tech/commerce hubs. The top 15 districts are all in Telangana, Karnataka, Maharashtra, and Tamil Nadu.

**3. Business Impact:**  
This tells merchant acquisition teams exactly which districts to focus on first. Concentrating business development resources in these top districts will yield the highest ROI.


#### Chart - 13: State x Quarter Transaction Heatmap (Heatmap)

In [ ]:
# Chart 13 - Heatmap of transaction counts: states vs year
# picking top 12 states by volume to keep the chart readable
top12_states = state_summary.head(12)['state'].tolist()
heatmap_data = (df_agg_txn[df_agg_txn['state'].isin(top12_states)]
    .groupby(['state', 'year'])['count']
    .sum()
    .unstack(fill_value=0)
)
heatmap_data_norm = heatmap_data.div(heatmap_data.max(axis=1), axis=0)  # normalize per state

fig, ax = plt.subplots(figsize=(11, 7))
sns.heatmap(heatmap_data_norm, cmap='YlOrRd', annot=False,
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Relative Volume (normalized per state)'})
ax.set_title('Transaction Volume Heatmap — Top States by Year', fontsize=13, fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('State')
plt.tight_layout()
plt.show()


**1. Why this chart?**  
A heatmap is the best way to compare 12 states across 6 years simultaneously. Normalizing per state shows relative growth patterns rather than absolute size differences.

**2. Insights:**  
Every state shows increasing color intensity year-on-year — confirming universal growth. Some states (like Telangana) were bright red even in 2019, while others only lit up from 2021-22, which tracks with their internet infrastructure rollout.

**3. Business Impact:**  
States that turned bright only recently are in an early-growth phase — competitive dynamics there are still fluid, making them ideal for aggressive promotions to lock in users before competitors do.


#### Chart - 14: Correlation Heatmap

In [ ]:
# Chart 14 - Correlation heatmap on the main numeric columns
# build a combined table: state/year/quarter with txn stats + user stats
txn_agg = (df_agg_txn
    .groupby(['state', 'year', 'quarter'], as_index=False)
    .agg(total_count=('count', 'sum'), total_amount=('amount', 'sum'))
)
user_agg = (df_agg_user
    .drop_duplicates(subset=['state', 'year', 'quarter'])
    [['state', 'year', 'quarter', 'registered_users', 'app_opens']]
)
combined = txn_agg.merge(user_agg, on=['state', 'year', 'quarter'], how='inner')
combined['avg_txn_value'] = combined['total_amount'] / combined['total_count'].replace(0, np.nan)

corr = combined[['total_count', 'total_amount', 'registered_users', 'app_opens', 'avg_txn_value']].corr()

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)  # show lower triangle only
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            linewidths=0.5, square=True, ax=ax,
            cbar_kws={'shrink': 0.8})
ax.set_title('Correlation Matrix — Key Metrics', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


**1. Why this chart?**  
The correlation heatmap reveals linear relationships between all numeric variables at once — it's a standard part of EDA to check which metrics move together.

**2. Insights:**  
Transaction count and registered users are very strongly correlated, which makes intuitive sense. App opens are highly correlated with both — engaged users transact more. Average transaction value shows a weak negative correlation with count, confirming the earlier finding that as volume grows, average ticket size shrinks.


#### Chart - 15: Pair Plot

In [ ]:
# Chart 15 - Pair plot for the combined numeric features
# sampling to keep it fast
sample = combined[['total_count', 'total_amount', 'registered_users', 'app_opens']].sample(
    min(2000, len(combined)), random_state=42
)
# rename for cleaner labels
sample.columns = ['Count', 'Amount', 'Reg. Users', 'App Opens']

pair_plot = sns.pairplot(sample, diag_kind='kde', plot_kws={'alpha': 0.3, 's': 15},
                          diag_kws={'fill': True})
pair_plot.fig.suptitle('Pair Plot — Key Transaction & User Metrics', y=1.02, fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


**1. Why this chart?**  
A pair plot gives a comprehensive view of all pairwise relationships and individual distributions in one figure — ideal for understanding the data space before any modeling.

**2. Insights:**  
The distributions are right-skewed for all variables — a few large states/quarters dominate. The amount vs count scatter shows a tight linear relationship (as expected). The KDE diagonals confirm the skewness and also show that the data is not normally distributed.


#### Chart - 16: Top 10 States by Insurance Premium Volume (Bar)

In [ ]:
# Chart 16 - Insurance: which states are buying the most coverage
ins_state = (df_agg_ins
    .groupby('state', as_index=False)
    .agg(total_amount=('amount', 'sum'), total_count=('count', 'sum'))
    .sort_values('total_amount', ascending=False)
    .head(10)
)

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.bar(ins_state['state'], ins_state['total_amount'] / 1e7,
              color=sns.color_palette('flare', 10), edgecolor='white')
ax.bar_label(bars, fmt='₹%.0f Cr', padding=3, fontsize=9)
ax.set_xlabel('State')
ax.set_ylabel('Insurance Premium Amount (₹ Crores)')
ax.set_title('Top 10 States by Insurance Transaction Value', fontsize=13, fontweight='bold')
plt.xticks(rotation=30, ha='right')
sns.despine()
plt.tight_layout()
plt.show()


**1. Why this chart?**  
Simple bar to compare a growing revenue vertical across states.

**2. Insights:**  
Maharashtra, Karnataka, and Tamil Nadu lead insurance premium volume — states with higher income levels and stronger awareness about financial products. There's a big drop-off after the top 3, suggesting insurance is still concentrated in a few states.

**3. Business Impact:**  
Geographic concentration of insurance revenue is a risk. Expanding insurance products to mid-tier states through partnerships with regional insurers or micro-insurance products could diversify revenue and reach new demographics.


#### Chart - 17: Transaction Count vs Amount by Type (Bubble Chart)

In [ ]:
# Chart 17 - Bubble chart: each transaction type plotted by count vs amount
type_summary = (df_agg_txn
    .groupby('transaction_type', as_index=False)
    .agg(total_count=('count', 'sum'), total_amount=('amount', 'sum'))
)
type_summary['avg_value'] = type_summary['total_amount'] / type_summary['total_count']

fig, ax = plt.subplots(figsize=(9, 6))
scatter = ax.scatter(
    type_summary['total_count'] / 1e7,
    type_summary['total_amount'] / 1e7,
    s=type_summary['avg_value'] / 50,   # bubble size = avg transaction value
    c=range(len(type_summary)),
    cmap='tab10', alpha=0.8, edgecolors='white', linewidth=1.5
)
for _, row in type_summary.iterrows():
    ax.annotate(row['transaction_type'],
                (row['total_count']/1e7, row['total_amount']/1e7),
                textcoords='offset points', xytext=(8, 4), fontsize=8.5)

ax.set_xlabel('Total Count (Crores)')
ax.set_ylabel('Total Amount (₹ Crores)')
ax.set_title('Transaction Count vs Amount by Type
(bubble size = avg transaction value)', fontsize=12, fontweight='bold')
sns.despine()
plt.tight_layout()
plt.show()


**1. Why this chart?**  
A bubble chart encodes three variables — count, amount, and average value — in a single visual. Much more informative than a simple bar for multi-dimensional comparison.

**2. Insights:**  
Financial Services has a tiny count but a large bubble (high average transaction value), confirming it's a low-frequency high-value category. P2P has the highest count. Merchant payments cluster in the moderate range for both count and amount — a balanced category.

**3. Business Impact:**  
Financial Services, despite low count, likely drives significant revenue per transaction. Growing this category — even marginally in count — has outsized revenue impact.


#### Chart - 18: Year-on-Year Growth Rate in Transaction Volume (Bar + Line)

In [ ]:
# Chart 18 - YoY growth rate per year
yearly_vol = df_agg_txn.groupby('year')['count'].sum().reset_index()
yearly_vol['yoy_growth'] = yearly_vol['count'].pct_change() * 100

fig, ax1 = plt.subplots(figsize=(9, 5))
ax2 = ax1.twinx()

bars = ax1.bar(yearly_vol['year'].astype(str), yearly_vol['count'] / 1e7,
               color='steelblue', alpha=0.6, width=0.4, label='Count (Cr)')
ax2.plot(yearly_vol['year'].astype(str), yearly_vol['yoy_growth'],
         color='red', marker='o', linewidth=2, markersize=7, label='YoY Growth %')
ax2.axhline(0, color='gray', linewidth=0.8, ls='--')

ax1.set_xlabel('Year')
ax1.set_ylabel('Transaction Count (Crores)', color='steelblue')
ax2.set_ylabel('YoY Growth Rate (%)', color='red')
ax1.set_title('Transaction Volume & Year-on-Year Growth Rate', fontsize=13, fontweight='bold')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
sns.despine()
plt.tight_layout()
plt.show()


**1. Why this chart?**  
Combining bar (absolute volume) with line (growth rate) on dual axes gives the full picture: how fast we're growing and how that rate is changing.

**2. Insights:**  
Growth rates were extremely high in 2019-20 as UPI was hitting mainstream. By 2022-23, growth has slowed to a more moderate pace — not because the market is shrinking, but because the base is now much larger. This is a natural S-curve dynamic.

**3. Business Impact:**  
Slowing growth rate on a large base is fine, but stakeholders should be cautious about projecting the same percentage growth forward. New user segments (rural, elderly, feature phone users) need targeted strategies to maintain high absolute growth.


#### Chart - 19: Top 15 Pin Codes by Transaction Count (Bar)

In [ ]:
# Chart 19 - Top 15 pincodes by transaction count
pincode_top = (df_top_txn[df_top_txn['level'] == 'pincodes']
    .groupby(['state', 'entity'], as_index=False)['count']
    .sum()
    .sort_values('count', ascending=False)
    .head(15)
)
pincode_top['label'] = pincode_top['entity'] + '\n' + pincode_top['state'].str[:6]

fig, ax = plt.subplots(figsize=(13, 5))
bars = ax.bar(pincode_top['label'], pincode_top['count'] / 1e5,
              color=sns.color_palette('Spectral_r', 15), edgecolor='white')
ax.set_xlabel('Pin Code (State)')
ax.set_ylabel('Transaction Count (Lakhs)')
ax.set_title('Top 15 Pin Codes by Transaction Count', fontsize=13, fontweight='bold')
plt.xticks(fontsize=8)
sns.despine()
plt.tight_layout()
plt.show()


**1. Why this chart?**  
Pin code level analysis is the most granular view available — useful for hyperlocal targeting.

**2. Insights:**  
The top pincodes are concentrated in Bengaluru, Hyderabad, and Chennai tech corridors. These are areas with high IT sector employment and younger demographics who adopted UPI early and use it for everything.

**3. Business Impact:**  
These high-density pincodes are ideal for piloting new features — the users there are tech-savvy, have high transaction frequency, and are likely to give quick feedback. Hyperlocal offers and rewards in these pincodes could have outsized engagement effects.


#### Chart - 20: App Opens vs Registered Users — Growth Quadrant Analysis (Scatter)

In [ ]:
# Chart 20 - Quadrant scatter: state-level registered users vs app opens
# categorizing states into quadrants based on median split
median_users = user_state['total_users'].median()
median_opens = user_state['total_opens'].median()

def get_quadrant(row):
    if row['total_users'] >= median_users and row['total_opens'] >= median_opens:
        return 'High Users + High Engagement'
    elif row['total_users'] >= median_users and row['total_opens'] < median_opens:
        return 'High Users + Low Engagement'
    elif row['total_users'] < median_users and row['total_opens'] >= median_opens:
        return 'Low Users + High Engagement'
    else:
        return 'Low Users + Low Engagement'

user_state['quadrant'] = user_state.apply(get_quadrant, axis=1)

colors_map = {
    'High Users + High Engagement'  : '#2ecc71',
    'High Users + Low Engagement'   : '#e67e22',
    'Low Users + High Engagement'   : '#3498db',
    'Low Users + Low Engagement'    : '#e74c3c'
}

fig, ax = plt.subplots(figsize=(11, 7))
for quad, group in user_state.groupby('quadrant'):
    ax.scatter(group['total_users']/1e6, group['total_opens']/1e6,
               label=quad, color=colors_map[quad], s=90, alpha=0.85, edgecolors='white')
    for _, row in group.iterrows():
        ax.annotate(row['state'], (row['total_users']/1e6, row['total_opens']/1e6),
                    textcoords='offset points', xytext=(4, 3), fontsize=7)

ax.axvline(median_users/1e6, ls='--', color='gray', linewidth=1)
ax.axhline(median_opens/1e6, ls='--', color='gray', linewidth=1)
ax.set_xlabel('Registered Users (Millions)')
ax.set_ylabel('App Opens (Millions)')
ax.set_title('State Segmentation: Users vs Engagement Quadrant', fontsize=13, fontweight='bold')
ax.legend(loc='upper left', fontsize=9)
sns.despine()
plt.tight_layout()
plt.show()


**1. Why this chart?**  
A quadrant analysis is a classic business tool for segmenting entities — here it helps identify which states are stars (high users, high engagement) and which ones need intervention.

**2. Insights:**  
Maharashtra and Karnataka are in the top-right quadrant — large and engaged. Several north Indian states fall in the lower quadrants — large populations but low UPI penetration and low engagement. States in the blue quadrant (low users, high engagement per user) are interesting — smaller but very sticky user bases.

**3. Business Impact:**  
Each quadrant needs a different strategy: Stars → retain and upsell; High Users Low Engagement → re-engagement campaigns; Low Users High Engagement → scale what's working; Low Low → ground-up UPI awareness programs. This chart directly feeds into budget allocation decisions.


## 5. Solution to Business Objective

#### What do you suggest to achieve the business objective?

Based on the analysis, here's what I'd recommend to the PhonePe product and business team:

**1. Double down on merchant payments in high-volume states.** Telangana, Karnataka, and Maharashtra are already mature markets for P2P. The next phase should convert those users into regular merchant payment users — which is both stickier and more monetizable.

**2. Target the user base in UP, Bihar, and MP aggressively.** These are massive-population states with relatively low UPI penetration. The data shows that once digital payments cross a certain adoption threshold in a state, growth becomes self-sustaining (network effects). Localized content, regional language support, and agent-based onboarding can help.

**3. Invest in Financial Services and Insurance.** These are low-count but high-value categories. Even a 20% volume increase in Financial Services transactions will have a disproportionate revenue impact. In-app contextual nudges (e.g., showing insurance options after a large payment) can help.

**4. Prioritize app performance on Xiaomi and Samsung mid-range devices.** The majority of users are on these devices. Any friction — slow load, UI glitches — directly impacts the largest user segment.

**5. Build hyperlocal strategies for top pin codes.** The top 15 pincodes are already heavily engaged. Launching premium features (UPI credit, investment products) there first will give faster feedback loops and higher activation rates.


# Conclusion

This EDA of the PhonePe Pulse dataset revealed several clear patterns in India's UPI payment landscape:

- **Growth has been consistent and steep** — every metric has grown year-on-year without exception, with the COVID-19 period acting as a major accelerator.
- **Geographic concentration is real** — Telangana, Maharashtra, and Karnataka dominate both count and value. The South and West are far ahead of the North and East.
- **P2P leads by count, but merchant and financial services lead by value** — the transaction mix is shifting toward merchant payments over time.
- **Average transaction value is declining** — a sign of healthy mass-market penetration where users are making small everyday payments through UPI.
- **Insurance is a rapidly growing vertical** with high premium values per transaction — significant revenue potential.
- **Device ecosystem is concentrated** on Xiaomi, Samsung, and Vivo — app optimization for these devices is critical.
- **The engagement quadrant analysis** clearly segments states into actionable buckets, giving the team a clear geographic strategy.

The dataset is clean, granular, and rich — there's a lot more analysis possible (time-series forecasting, district-level regression, etc.) that can build on this foundation.

---
### Hurrah! EDA Capstone completed successfully 🎉
